# VascularAge — Phase 5B / Compensation Tie-Sensitivity Closure

This notebook closes the prospectively required co-nearest-target sensitivity using **preserved Phase-4 evidence only**. It does not access PWDB waveforms, VascuQuest, a GPU, or any random generator.

Phase-5B lock: `6b9f11bf0c662c8263e531b0440882881d79b5ab70aaa4aaeffd4e4a69d741c2`

Run all cells unchanged. A PASS establishes that every P0-aliased source has a unique co-nearest target under the locked `1e-6` tolerance.


In [1]:
from pathlib import Path
import json, subprocess, sys
from google.colab import drive

BRANCH = "phase-05b-tie-sensitivity-closure"
LOCK_SHA = "6b9f11bf0c662c8263e531b0440882881d79b5ab70aaa4aaeffd4e4a69d741c2"
drive.mount("/content/drive")

REPO = Path("/content/VascularAge")
subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, "https://github.com/khalid-saqr/VascularAge.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test]"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=True)

va_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
manifest = json.loads((REPO / "phase5b" / "TIE_SENSITIVITY_LOCK.json").read_text())
assert manifest["tie_sensitivity_lock_sha256"] == LOCK_SHA
print("VascularAge:", va_commit)
print("Phase-5B tie-sensitivity lock:", LOCK_SHA)


Mounted at /content/drive
VascularAge: 2f38433078342f8418a480fce027341413ac8a8a
Phase-5B tie-sensitivity lock: 6b9f11bf0c662c8263e531b0440882881d79b5ab70aaa4aaeffd4e4a69d741c2


In [2]:
cmd = [
    sys.executable,
    str(REPO / "scripts" / "phase5b_tie_closure.py"),
    "--execute-tie-closure",
    "--expected-lock-sha", LOCK_SHA,
    "--repo-root", str(REPO),
    "--phase4-evidence-root", "/content/drive/MyDrive/VascularAge/phase_04/locked_trial_amendment001_20260829T153743Z",
    "--output-parent", "/content/drive/MyDrive/VascularAge/phase_05b",
]
proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
assert "PHASE 5B TIE-SENSITIVITY CLOSURE: PASS" in proc.stdout
assert "PHASE 5B CLOSURE: COMPLETE" in proc.stdout
print("PHASE 5B NOTEBOOK: SUCCESS")


PHASE 5B TIE-SENSITIVITY CLOSURE: PASS
{
  "P0_alias_sources": 2764,
  "P0_alias_sources_with_co_nearest_alternatives": 0,
  "S4_reopened": false,
  "canonical_top20_motif_concentration": 0.5821273516642547,
  "closure_adjudication": "PASS",
  "co_nearest_sensitive_top20_motif_concentration": 0.5821273516642547,
  "endpoint": "COMPENSATION_TIE_SENSITIVITY_CLOSURE",
  "locked_null_95th": 0.033646888567293774,
  "maximum_co_nearest_count_P0_alias": 1,
  "maximum_co_nearest_count_all": 1,
  "minimum_nearest_second_gap_P0_alias": 4.976987838745117e-05,
  "minimum_nearest_second_gap_all": 1.4781951904296875e-05,
  "new_null_distribution_generated": false,
  "outcome": "NO_CO_NEAREST_ALTERNATIVES",
  "phase": "5B",
  "randomness_used": false,
  "status": "EXECUTED",
  "subjects": 4374,
  "subjects_with_co_nearest_alternatives_all": 0,
  "tie_atol": 1e-06,
  "tie_sensitivity_lock_sha256": "6b9f11bf0c662c8263e531b0440882881d79b5ab70aaa4aaeffd4e4a69d741c2",
  "top20_difference": 0.0,
  "unorder